# 📝 Day 3 Assignments — Pydantic v2

---

Four tasks plus a bonus. Each task pushes a little deeper into validation: constraints → endpoints → nested models → custom validators.


In [ ]:
!pip install pydantic fastapi uvicorn email-validator


## Task 1 — User model with constraints

**Problem:** Build a `User` model with:
- `name`: string, 3–50 chars
- `age`: int, 0–150
- `email`: use `EmailStr` if available, otherwise a regex pattern

Then instantiate one **valid** user and catch a `ValidationError` on an invalid one.

**Expected output:**
```
VALID:    name=... age=... email=...
INVALID: <list of field errors>
```

💡 **Hint:** `from pydantic import EmailStr`. If the package isn't installed, fall back to `Field(..., pattern=r"...")`.


In [2]:
from pydantic import BaseModel, Field, ValidationError

try:
    from pydantic import EmailStr
    EmailType = EmailStr
except ImportError:
    # Fallback if email-validator isn't installed — basic regex
    EmailType = str

class User(BaseModel):
    name: str = Field(..., min_length=3, max_length=50)
    age: int = Field(..., ge=0, le=150)
    email: EmailType = Field(..., pattern=r"^[^@\s]+@[^@\s]+\.[^@\s]+$")

print("VALID:  ", User(name="Alice", age=30, email="alice@example.com"))

try:
    User(name="Al", age=200, email="not-an-email")
except ValidationError as e:
    print("\nINVALID:")
    for err in e.errors():
        print(" ", err["loc"], "->", err["msg"])


ImportError: email-validator is not installed, run `pip install 'pydantic[email]'`

## Task 2 — `POST /users` endpoint

**Problem:** Wrap the `User` model in a FastAPI endpoint. Send both a valid and an invalid request with `TestClient`; show that the invalid one returns **422**.

💡 **Hint:** if FastAPI sees `user: User`, it validates automatically.


In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI()

@app.post("/users")
def create_user(user: User):
    return {"created": user.model_dump()}

client = TestClient(app)

ok = client.post("/users", json={"name": "Alice", "age": 30, "email": "alice@x.com"})
print("VALID  ->", ok.status_code, ok.json())

bad = client.post("/users", json={"name": "Al", "age": 999, "email": "nope"})
print("INVALID->", bad.status_code, len(bad.json()["detail"]), "errors")


## Task 3 — Nested `Order` model

**Problem:** Build:

```
OrderItem { product: str, qty: int > 0, price: float > 0 }
Order     { items: list[OrderItem], total: float >= 0 }
```

Create one valid `Order` and one that fails validation (e.g. qty 0 or total negative).

💡 **Hint:** Pydantic walks nested models automatically — the failing field's `loc` will be a tuple like `('items', 0, 'qty')`.


In [ ]:
class OrderItem(BaseModel):
    product: str = Field(..., min_length=1)
    qty:     int   = Field(..., gt=0)
    price:   float = Field(..., gt=0)

class Order(BaseModel):
    items: list[OrderItem]
    total: float = Field(..., ge=0)

order = Order(
    items=[
        {"product": "Pen",  "qty": 2, "price": 2.5},
        {"product": "Book", "qty": 1, "price": 12.0},
    ],
    total=17.0,
)
print("VALID:", order)

try:
    Order(
        items=[{"product": "Pen", "qty": 0, "price": 2.5}],
        total=-1.0,
    )
except ValidationError as e:
    print("\nINVALID:")
    for err in e.errors():
        print(" ", err["loc"], "->", err["msg"])


## Task 4 — `@field_validator` for passwords

**Problem:** Build an `Account` model with `username` and `password`. Use `@field_validator("password")` to enforce:

- at least 8 characters
- must contain at least one digit

**Expected output:** valid signup goes through; bad password raises `ValidationError` with your custom messages.

💡 **Hint:** validators return the value on success and `raise ValueError(...)` on failure.


In [3]:
from pydantic import field_validator

class Account(BaseModel):
    username: str = Field(..., min_length=3)
    password: str

    @field_validator("password")
    @classmethod
    def strong_enough(cls, v: str) -> str:
        if len(v) < 8:
            raise ValueError("password must be at least 8 characters")
        if not any(c.isdigit() for c in v):
            raise ValueError("password must contain at least one digit")
        return v

print("VALID:  ", Account(username="alice", password="hunter22"))

for bad_pw in ["short1", "nodigits!"]:
    try:
        Account(username="alice", password=bad_pw)
    except ValidationError as e:
        print(f"\n{bad_pw!r} ->", e.errors()[0]["msg"])


VALID:   username='alice' password='hunter22'

'short1' -> Value error, password must be at least 8 characters

'nodigits!' -> Value error, password must contain at least one digit


## 🎁 Bonus — `model_config` + `@model_validator`

**Problem:** Build a `Booking` model with `start_date` and `end_date` (both `date`).

- Use `model_config = ConfigDict(str_strip_whitespace=True)` so string fields get trimmed automatically.
- Use `@model_validator(mode="after")` to ensure `end_date > start_date`.

💡 **Hint:** `model_validator(mode="after")` runs **after** field validation and receives `self`.


In [ ]:
from datetime import date
from pydantic import ConfigDict, model_validator

class Booking(BaseModel):
    model_config = ConfigDict(str_strip_whitespace=True)

    guest_name: str = Field(..., min_length=1)
    start_date: date
    end_date: date

    @model_validator(mode="after")
    def dates_make_sense(self) -> "Booking":
        if self.end_date <= self.start_date:
            raise ValueError("end_date must be after start_date")
        return self

# Whitespace gets stripped, dates ordered correctly → valid
b = Booking(guest_name="  Alice  ", start_date="2026-06-01", end_date="2026-06-05")
print("VALID:", b)
print("guest_name stripped ->", repr(b.guest_name))

try:
    Booking(guest_name="Bob", start_date="2026-06-10", end_date="2026-06-01")
except ValidationError as e:
    print("\nINVALID:", e.errors()[0]["msg"])


---

✅ **Done!** You can now validate any incoming JSON at the edge of your API — from simple constraints to whole-model invariants.
